# Module 06 — Lecture 2: cuFFT for Neural Signal Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_06_advanced_topics/02_cufft_signal_analysis.ipynb)

---

Local field potentials (LFPs), EEG, and MEG recordings require spectral analysis to identify oscillatory bands (theta 4–8 Hz, alpha 8–12 Hz, beta 12–30 Hz, gamma 30–100 Hz). With 64–256 electrode arrays and hours of recording, this quickly becomes a GPU job.

**Learning objectives:**
- Compute power spectra with cuFFT on real-scale electrode array data
- Build a Short-Time Fourier Transform (STFT) spectrogram pipeline
- Compute coherence between two LFP channels
- Understand when cuFFT outperforms CPU NumPy FFT

In [ ]:
!nvidia-smi

## 1. cuFFT API

cuFFT mirrors FFTW's planning model:

```c
cufftHandle plan;

// Plan: N_channels 1D transforms of length N_samples
cufftPlan1d(&plan, N_samples, CUFFT_R2C, N_channels);

// Execute (input: float*, output: cufftComplex*)
cufftExecR2C(plan, d_signal, d_spectrum);

// Destroy when done
cufftDestroy(plan);
```

**Real-to-Complex (R2C):** Input N real samples → output N/2+1 complex frequency bins. The output is Hermitian-symmetric so only the positive frequencies are stored.

**Batch dimension:** The batch size in `cufftPlan1d` processes multiple independent signals simultaneously — perfect for multi-electrode arrays.

**Normalization:** cuFFT does NOT normalize. Divide by N to get the standard DFT convention.

In [ ]:
# First: generate synthetic multi-channel LFP and run the full pipeline
%%writefile lfp_analysis.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cufft.h>

#define CUDA_CHECK(call) do { cudaError_t e=(call); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__, \
    cudaGetErrorString(e));exit(1);}} while(0)
#define CUFFT_CHECK(call) do { cufftResult e=(call); \
    if(e!=CUFFT_SUCCESS){fprintf(stderr,"cuFFT error %d\n",e);exit(1);}} while(0)

// Hann window in-place
__global__ void hann_window(float* x, int N, int n_ch) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= n_ch * N) return;
    int s = idx % N;
    x[idx] *= 0.5f * (1.0f - cosf(2.0f * M_PI * s / (N - 1)));
}

// Power spectrum: |FFT[k]|^2 / N^2
__global__ void power_spectrum(const cufftComplex* X, float* P,
                                int N_fft, int n_ch) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int n_freq = N_fft / 2 + 1;
    if (idx >= n_ch * n_freq) return;
    cufftComplex c = X[idx];
    P[idx] = (c.x*c.x + c.y*c.y) / ((float)N_fft * N_fft);
}

// Cross-spectrum: X1[k] * conj(X2[k])
__global__ void cross_spectrum(const cufftComplex* X1, const cufftComplex* X2,
                                cufftComplex* Pxy, int n_freq) {
    int k = blockIdx.x * blockDim.x + threadIdx.x;
    if (k >= n_freq) return;
    Pxy[k].x = X1[k].x * X2[k].x + X1[k].y * X2[k].y;  // Re(X1 * conj(X2))
    Pxy[k].y = X1[k].y * X2[k].x - X1[k].x * X2[k].y;  // Im(X1 * conj(X2))
}

int main(int argc, char** argv) {
    float fs   = 1000.f;
    float T    = 10.f;
    int   N    = (int)(fs * T);
    int   n_ch = (argc>1) ? atoi(argv[1]) : 64;
    int   n_freq = N / 2 + 1;

    printf("LFP analysis: %d channels × %d samples @ %.0f Hz\n", n_ch, N, fs);

    // Generate synthetic LFP: theta (8 Hz) + gamma (40 Hz) + noise
    float* h_lfp = (float*)malloc((size_t)n_ch * N * sizeof(float));
    srand(42);
    for (int ch = 0; ch < n_ch; ch++) {
        float ph_th = (float)rand()/RAND_MAX * 2*M_PI;
        float ph_gm = (float)rand()/RAND_MAX * 2*M_PI;
        for (int t = 0; t < N; t++) {
            float sec = (float)t / fs;
            float noise = 0.5f * ((float)rand()/RAND_MAX*2.f - 1.f);
            h_lfp[ch*N + t] = 2.f*sinf(2*M_PI*8.f*sec+ph_th)
                             + 1.f*sinf(2*M_PI*40.f*sec+ph_gm) + noise;
        }
    }

    float *d_lfp;
    cufftComplex *d_fft;
    float *d_power;
    CUDA_CHECK(cudaMalloc(&d_lfp,  (size_t)n_ch*N*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_fft,  (size_t)n_ch*N*sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_power,(size_t)n_ch*n_freq*sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_lfp, h_lfp, (size_t)n_ch*N*sizeof(float),
                          cudaMemcpyHostToDevice));

    int thr=256;
    int blk_sig  = ((size_t)n_ch*N + thr-1)/thr;
    int blk_freq = ((size_t)n_ch*n_freq + thr-1)/thr;

    hann_window<<<blk_sig, thr>>>(d_lfp, N, n_ch);

    cufftHandle plan;
    CUFFT_CHECK(cufftPlan1d(&plan, N, CUFFT_R2C, n_ch));

    cudaEvent_t t0,t1; float ms;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));

    int REPS=50;
    CUDA_CHECK(cudaEventRecord(t0));
    for (int r=0;r<REPS;r++) CUFFT_CHECK(cufftExecR2C(plan, d_lfp, d_fft));
    CUDA_CHECK(cudaEventRecord(t1)); CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&ms,t0,t1));
    printf("Batch FFT (%d × N=%d): %.3f ms/call\n", n_ch, N, ms/REPS);

    power_spectrum<<<blk_freq, thr>>>(d_fft, d_power, N, n_ch);

    // Copy back power spectrum of channel 0
    float* h_power = (float*)malloc(n_freq * sizeof(float));
    CUDA_CHECK(cudaMemcpy(h_power, d_power, n_freq*sizeof(float),
                          cudaMemcpyDeviceToHost));

    // Print dominant peaks
    float df = fs / N;
    printf("\nChannel 0 power spectrum peaks:\n");
    for (int b=2; b<n_freq-1; b++) {
        if (h_power[b]>h_power[b-1] && h_power[b]>h_power[b+1] &&
            h_power[b] > 0.001f)
            printf("  %.1f Hz: P=%.4f\n", b*df, h_power[b]);
    }

    // Save power spectrum for Python
    FILE* fp=fopen("lfp_power.txt","w");
    for (int b=0;b<n_freq;b++) fprintf(fp,"%.3f %.8e\n", b*df, h_power[b]);
    fclose(fp);

    // Coherence between ch 0 and ch 1
    cufftComplex *d_pxy; float *d_p1, *d_p2;
    CUDA_CHECK(cudaMalloc(&d_pxy, n_freq*sizeof(cufftComplex)));
    CUDA_CHECK(cudaMalloc(&d_p1,  n_freq*sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_p2,  n_freq*sizeof(float)));

    // d_fft[0..n_freq-1] = ch0, d_fft[N..N+n_freq-1] = ch1
    cross_spectrum<<<(n_freq+thr-1)/thr, thr>>>(
        d_fft, d_fft + N, d_pxy, n_freq);

    // Copy ch1 power
    power_spectrum<<<(n_freq+thr-1)/thr, thr>>>(d_fft+N, d_p2, N, 1);

    cufftComplex* h_pxy=(cufftComplex*)malloc(n_freq*sizeof(cufftComplex));
    float* h_p2=(float*)malloc(n_freq*sizeof(float));
    CUDA_CHECK(cudaMemcpy(h_pxy,d_pxy,n_freq*sizeof(cufftComplex),cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_p2, d_p2, n_freq*sizeof(float),        cudaMemcpyDeviceToHost));

    FILE* fc=fopen("lfp_coherence.txt","w");
    printf("\nCoherence ch0-ch1 at key frequencies:\n");
    for (int b=0;b<n_freq;b++) {
        float Pxy2 = h_pxy[b].x*h_pxy[b].x + h_pxy[b].y*h_pxy[b].y;
        float coh  = (h_power[b] > 1e-10f && h_p2[b] > 1e-10f)
                     ? Pxy2 / (h_power[b] * h_p2[b]) : 0.f;
        fprintf(fc, "%.3f %.6f\n", b*df, coh);
        float f=b*df;
        if (fabsf(f-8.f)<df || fabsf(f-40.f)<df)
            printf("  %.1f Hz: coherence=%.3f\n", f, coh);
    }
    fclose(fc);

    CUFFT_CHECK(cufftDestroy(plan));
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_lfp); cudaFree(d_fft); cudaFree(d_power);
    cudaFree(d_pxy); cudaFree(d_p1); cudaFree(d_p2);
    free(h_lfp); free(h_power); free(h_pxy); free(h_p2);
    return 0;
}

In [ ]:
!nvcc -O2 -o lfp_analysis lfp_analysis.cu -lcufft -lm && ./lfp_analysis 64

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load GPU power spectrum
ps_data = np.loadtxt('lfp_power.txt')
freqs, power = ps_data[:, 0], ps_data[:, 1]

# Load coherence
coh_data = np.loadtxt('lfp_coherence.txt')
coh_freqs, coherence = coh_data[:, 0], coh_data[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Power spectrum
f_max = 200
mask = freqs <= f_max
axes[0].semilogy(freqs[mask], power[mask], 'b-', lw=1.5)
axes[0].set_xlabel('Frequency (Hz)', fontsize=12)
axes[0].set_ylabel('Power (log scale)', fontsize=12)
axes[0].set_title('LFP Power Spectrum (GPU cuFFT)', fontsize=12)
for f, label in [(8, 'θ'), (40, 'γ')]:
    axes[0].axvline(f, color='r', linestyle='--', alpha=0.5)
    axes[0].text(f+1, power[mask].max()*0.5, label, color='r', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Coherence
coh_mask = coh_freqs <= f_max
axes[1].plot(coh_freqs[coh_mask], coherence[coh_mask], 'g-', lw=1.5)
axes[1].set_xlabel('Frequency (Hz)', fontsize=12)
axes[1].set_ylabel('Coherence', fontsize=12)
axes[1].set_title('LFP Coherence: ch0 vs ch1', fontsize=12)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

# CPU vs GPU FFT timing comparison
import time
import subprocess

n_channels_list = [1, 8, 32, 64, 128]
cpu_times, gpu_times = [], []
fs = 1000; T = 10; N = fs * T

for n_ch in n_channels_list:
    # CPU NumPy FFT
    sig = np.random.randn(n_ch, N).astype(np.float32)
    t_start = time.perf_counter()
    for _ in range(20):
        _ = np.fft.rfft(sig, axis=1)
    cpu_times.append((time.perf_counter() - t_start) / 20 * 1000)

    # GPU cuFFT
    result = subprocess.run(['./lfp_analysis', str(n_ch)],
                            capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if 'Batch FFT' in line:
            gpu_times.append(float(line.split()[3]))
            break

axes[2].plot(n_channels_list, cpu_times, 'r-o', lw=2, label='CPU NumPy FFT')
axes[2].plot(n_channels_list, gpu_times, 'b-s', lw=2, label='GPU cuFFT')
axes[2].set_xlabel('Number of channels', fontsize=12)
axes[2].set_ylabel('Time per call (ms)', fontsize=12)
axes[2].set_title('CPU vs GPU FFT Scaling', fontsize=12)
axes[2].legend(fontsize=10); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cufft_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. STFT Spectrogram Pipeline

The STFT splits the signal into overlapping windows and FFTs each one. This reveals how frequency content changes over time — critical for detecting transient oscillations (e.g., gamma bursts).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import subprocess

# Run the full cufft_lfp.cu program which outputs the spectrogram
# (assumes it was compiled earlier via src/run_local.sh)
# For Colab, compile inline:
!nvcc -O2 -o cufft_lfp ../src/cufft_lfp.cu -lcufft -lm 2>/dev/null || echo "Already compiled"
!./cufft_lfp

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load spectrogram data
with open('lfp_spectrogram.txt') as f:
    # Skip comment lines, read matrix
    lines = [l for l in f if not l.startswith('#')]

spectrogram = np.array([[float(x) for x in l.split()] for l in lines])
print(f"Spectrogram shape: {spectrogram.shape} (freq_bins × time_frames)")

fs = 1000; win = 256; hop = 64
N_total = 4000  # 4 seconds
n_frames = (N_total - win) // hop + 1
n_freq = win // 2 + 1

freq_bins = np.fft.rfftfreq(win, 1/fs)
time_frames = np.arange(n_frames) * hop / fs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full spectrogram
max_f = 150  # Hz
f_mask = freq_bins <= max_f
S_plot = np.log10(spectrogram[f_mask, :] + 1e-12)  # dB-like scale

im = axes[0].imshow(S_plot, aspect='auto', origin='lower',
                    extent=[time_frames[0], time_frames[-1],
                            freq_bins[0], freq_bins[f_mask].max()],
                    cmap='viridis')
plt.colorbar(im, ax=axes[0], label='log₁₀ Power')
axes[0].set_xlabel('Time (s)', fontsize=12)
axes[0].set_ylabel('Frequency (Hz)', fontsize=12)
axes[0].set_title('LFP Spectrogram (GPU cuFFT STFT)', fontsize=12)
for f, label in [(8, 'θ 8Hz'), (40, 'γ 40Hz'), (120, 'hγ 120Hz')]:
    if f <= max_f:
        axes[0].axhline(f, color='white', linestyle='--', alpha=0.5, lw=0.8)
        axes[0].text(time_frames[-1]*0.98, f+2, label, color='white',
                    fontsize=8, ha='right')

# Time-averaged spectrum from STFT
mean_power = spectrogram[f_mask, :].mean(axis=1)
axes[1].semilogy(freq_bins[f_mask], mean_power, 'b-', lw=2)
axes[1].set_xlabel('Frequency (Hz)', fontsize=12)
axes[1].set_ylabel('Mean Power', fontsize=12)
axes[1].set_title('Time-Averaged Spectrum from STFT', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lfp_spectrogram_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Task | cuFFT call | Notes |
|------|-----------|-------|
| Power spectrum (1 channel) | `cufftExecR2C` | Divide by N² for amplitude |
| Multi-channel batch | `cufftPlan1d(..., N_ch)` | All channels in one call |
| Coherence | Cross-spectrum + normalize | X1 × conj(X2) / √(P1 × P2) |
| Spectrogram | Loop STFT windows | Or use `cufftPlanMany` for overlap-add |

**When GPU wins:** For N_channels ≥ 8 and N_samples ≥ 1000, cuFFT substantially outperforms CPU NumPy. The crossover point depends on data transfer overhead (PCIe). For online analysis of live recordings, keep data resident on GPU.

**Next lecture:** Multi-GPU scaling for large networks.

---

**Next →** [03 — Multi-GPU Scaling](03_multi_gpu_scaling.ipynb) &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_06_advanced_topics/03_multi_gpu_scaling.ipynb)